# Literature Review: Measuring the 15-Minute City in Shanghai

The 15-minute city (15MC) reframes urban quality as a question of everyday proximity: residents should be able to reach core needs such as food, health care, education, public space, and social life within a short walk or cycle trip. Moreno et al. (2021) connect the idea to chrono-urbanism, a planning argument that cities should reduce the amount of time residents must spend moving between separated functions. The concept is not only about transport efficiency. It is also about place identity, local services, social resilience, and the everyday dignity of having useful destinations close to home. For this project, that idea is operationalised through a 500 m grid, mode-specific 15-minute access proxies, and an H3 aggregation layer that can be inspected in an interactive map.

Measurement is the main methodological challenge. A strict 15MC study would use network travel times, sidewalk quality, intersection delay, service capacity, opening hours, and local demand. Many public dashboards simplify the idea into distance to amenities, which is understandable but risky: straight-line access can exaggerate true walkability, while POI counts can reward quantity even when services are small, closed, unaffordable, or difficult to reach. Weng et al. (2019) are especially relevant for this Shanghai project because they measured 15-minute walkable neighbourhoods in urban China and argued that amenity access should be sensitive to population groups, real traffic conditions, amenity scale, and social inequality. Their approach shows that the 15MC can be measured empirically, but also that the result depends heavily on assumptions about what counts as access.

The supplied project data contains 2023 Shanghai point-of-interest shapefiles rather than routable street geometry, GTFS timetables, current opening hours, rental listings, or API credentials. This notebook therefore uses Euclidean mode buffers as a transparent proxy: 1.2 km walking, 3.5 km cycling, 6 km transit, and 10 km car. The method is weaker than a real network isochrone produced by Gaode, OpenRouteService, or a local street graph, but it is reproducible, fast enough for the class project, and explicit about its assumptions. The web application labels car access as comparison only, because the 15MC baseline in the brief is explicitly about walking and cycling rather than automobile reach.

The baseline indicators follow the brief's universal-needs logic: food, shopping, health care, education/culture, parks/public amenities, and transit access. Track B adds restaurants, bars/nightlife, cultural venues, convenience stores, and a metro-late proxy. Counts are transformed using a mixed adequacy and richness score. A single nearby venue gives basic credit because the resident can reach the service, while denser districts receive additional credit for choice and variety. This avoids treating one shop the same as a large commercial street, while also avoiding a map where central Shanghai becomes a flat surface of maximum scores. H3 resolution 8 is used because it creates legible neighbourhood-scale hexagons while keeping the web payload manageable.

Equity critiques are central to the 15MC debate. Pozoukidou and Chatziyiannaki (2021) argue that the 15MC should be decomposed into measurable dimensions rather than treated as a universal slogan. Their framing matters because a polished map of access can hide whether services are inclusive, affordable, and socially useful. Khavarian-Garmsir et al. (2023) similarly review the concept's planning principles and implementation challenges, warning that 15MC policy can become a branding exercise if it does not confront housing affordability, governance capacity, service quality, and displacement. In high-demand cities, walkable amenity-rich districts can become expensive districts. A neighbourhood may therefore score well on proximity while excluding the residents who would most benefit from reduced travel burden.

For Shanghai nightlife, the key question is whether cultural and entertainment clusters are genuinely accessible to residents without relying on destination travel. Track B is a useful stress test because nightlife is not the same as daily-service access. A place can have schools, clinics, shops, and parks while still lacking evening social infrastructure. Conversely, a nightlife cluster can be attractive to visitors while offering weak everyday livability or high housing costs. The analysis expects high scores in dense central districts such as Huangpu, Jing'an, and Xuhui, with weaker coverage in peripheral areas. The important interpretation is not just where scores are high, but where baseline access and nightlife access diverge.

This version should be read as a reproducible analytical prototype rather than a definitive policy model. Real opening hours, Dianping ratings, review counts, late-night metro timetables, pedestrian network distances, bike-lane coverage, and rent data would all improve the analysis. The project still contributes a useful first pass: it documents the data provenance, translates the brief into a measurable scoring system, exposes limitations to users, and provides an interactive H3 map for exploring the geography of Shanghai's 15-minute nightlife access.

References:

- Moreno, C., Allam, Z., Chabaud, D., Gall, C., & Pratlong, F. (2021). Introducing the "15-Minute City": Sustainability, Resilience and Place Identity in Future Post-Pandemic Cities. Smart Cities, 4(1), 93-111. https://doi.org/10.3390/smartcities4010006
- Weng, M., Ding, N., Li, J., Jin, X., Xiao, H., He, Z., & Su, S. (2019). The 15-minute walkable neighborhoods: Measurement, social inequalities and implications for building healthy communities in urban China. Journal of Transport & Health, 13, 259-273.
- Pozoukidou, G., & Chatziyiannaki, Z. (2021). 15-Minute City: Decomposing the New Urban Planning Eutopia. Sustainability, 13(2), 928. https://doi.org/10.3390/su13020928
- Khavarian-Garmsir, A. R., Sharifi, A., & Sadeghi, A. (2023). The 15-minute city: Urban planning and design efforts toward creating sustainable neighborhoods. Cities, 132, 104101.

## Notebook Purpose

This notebook documents data sourcing, POI cleaning, category mapping, validation, and provenance. It runs the project script `src/01_collect_pois.py`, which converts the supplied 2023 Shanghai POI shapefiles into a clean CSV used by the later notebooks.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = ROOT / "2023_Shp" / "Shp"
PROCESSED = ROOT / "data" / "processed"
print(ROOT)
print(len(list(RAW_DIR.glob("*.shp"))), "raw shapefiles")

## Run Data Collection

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, str(ROOT / "src" / "01_collect_pois.py")], check=True)

## Validate Classified POIs

In [ ]:
poi_path = PROCESSED / "poi_catalog.csv"
summary_path = PROCESSED / "poi_summary.json"
pois = pd.read_csv(poi_path)
summary = json.loads(summary_path.read_text(encoding="utf-8"))

print(f"Classified POIs: {len(pois):,}")
display(pois.head())
summary["by_indicator"]

In [ ]:
district_counts = pois["district"].value_counts().head(16)
district_counts

## Data Provenance and Limitations

- Source: supplied `2023_Shp/Shp/*.shp` point shapefiles, WGS84 coordinates.
- Key fields used: `name`, `type`, `adname`, `经度`, `纬度`, `timestamp`.
- Category mapping: deterministic keyword rules in `src/config.py`.
- Address text is not used for category matching, because it can contain phrases such as "near metro station" that misclassify unrelated POIs.
- Known limitations: duplicated POIs, possible closed venues, no opening hours, no ratings, and no platform terms beyond the local supplied files.